# Pipeline CD - Coleta de Dados
## Trilha C: risco de baixa produtividade agrícola
### ETAPA 1 - Coleta e preservação do dado bruto

1. **Configuração**: parâmetros de local, período e variáveis reunidos em um único lugar;
2. **Coleta**: requisições às duas APIs da Trilha C (NASA POWER e IBGE/PAM), com timeout e tratamento de erro;
3. **Preservação do dado bruto**: a resposta original de cada API é salva em disco antes de qualquer transformação.



## 1. Definição do problema


- **Evento a prever:** Risco de baixa produtividade da lavoura de soja em Piracicaba-SP.
- **Horizonte:** por safra (uma linha = uma safra de soja no município).
- **Recorte geográfico:** Piracicaba-SP (código IBGE 3538709).
- **Período:** safras 2015 a 2025.
- **Unidade de análise:** ano-safra de soja em Piracicaba-SP.
- **Informações disponíveis no momento real da previsão:** dados climáticos da janela de plantio/desenvolvimento (Setembro do ano anterior a Abril do ano da safra) variáveis agrícolas do IBGE (área, quantidade, rendimento) só existem **depois** da colheita e por isso não podem virar feature, apenas alvo.
- **Custo do falso negativo:** não prever uma safra de baixa produtividade significa que o produtor não antecipa medidas de prevenção (irrigação, ajuste de insumos, seguro agrícola), aumentando o prejuízo financeiro quando o evento realmente ocorre.

In [1]:
import json
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

## 2. Configuração da coleta

Todos os parâmetros usados nas duas coletas 

In [2]:
from pathlib import Path
# Identifica a pasta data/raw independente se executado da raiz ou de dentro de notebooks/
PASTA_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_RAW = PASTA_RAIZ / "data" / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)

config = {
    "local": "Piracicaba-SP",
    "latitude": -22.7253,
    "longitude": -47.6492,
    "data_inicial": "2015-01-01",
    "data_final": "2025-12-31",
    "ano_inicial": 2015,
    "ano_final": 2025,
    "timezone": "America/Sao_Paulo",
    "cultura": "Soja (em grão)",
}




## 3. Fontes escolhidas

Conforme a Tabela de Decisão Ponderada da ficha de investigação (IBGE = 22 pontos, NASA POWER = 21 pontos):

| Fonte | Papel no projeto | Justificativa |
|---|---|---|
| **IBGE — API de Dados Agregados (PAM, tabela 1612)** | Fornece a variável-alvo (rendimento médio, kg/ha) e atributos de área/produção | Única fonte com o dado de produtividade por município e cultura |
| **NASA POWER Daily API** | Fornece as variáveis climáticas explicativas | Cobertura temporal mais longa e melhor compatibilidade com as demais fontes que o Open-Meteo (usado aqui apenas como fallback documentado) |

WHO/OMS e WMO/OMM **não** entram como coleta — são usadas só para fundamentar limiares e interpretar resultados na ETAPA 2.

## 4. Coleta 1 — Clima agrícola (NASA POWER Daily API)

Variáveis selecionadas — cada uma com justificativa ligada ao problema (registradas no dicionário de dados do grupo):

| Variável | Justificativa |
|---|---|
| `PRECTOTCORR` | precipitação: indicador direto de déficit/excesso hídrico na safra |
| `T2M` | temperatura média: estresse térmico geral da cultura |
| `T2M_MAX` | temperatura máxima: dias de calor extremo |
| `T2M_MIN` | temperatura mínima: risco de frio/geada no início do ciclo |
| `RH2M` | umidade relativa: combinada com temperatura para déficit hídrico |
| `GWETROOT` | umidade do solo na zona radicular: proxy direto de seca agrícola |

In [10]:
url_clima_agricola = "https://power.larc.nasa.gov/api/temporal/daily/point"

variaveis_climaticas = [
    "PRECTOTCORR",
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "GWETROOT",
]

parametros_clima_agricola = {
    "parameters": ",".join(variaveis_climaticas),
    "community": "AG",  # Agroclimatology: unidades e variáveis voltadas à agricultura
    "longitude": config["longitude"],
    "latitude": config["latitude"],
    "start": config["data_inicial"].replace("-", ""),
    "end": config["data_final"].replace("-", ""),
    "format": "JSON",
}

try:
    resposta_clima_agricola = requests.get(
        url_clima_agricola,
        params=parametros_clima_agricola,
        timeout=60,
    )
    resposta_clima_agricola.raise_for_status()
    dados_clima_agricola = resposta_clima_agricola.json()
    print("Requisição concluída.")
except requests.RequestException as erro:
    print("Falha na requisição:", erro)
    dados_clima_agricola = None

Requisição concluída.


### 4.1 Preservar o dado bruto

O nome do arquivo é determinístico (baseado na `config`, não em timestamp): rodar a coleta de novo **sobrescreve** o mesmo arquivo em vez de duplicar registros.

In [11]:
if "dados_clima_agricola" not in globals() or dados_clima_agricola is None:
    raise NameError("Execute a célula de coleta do clima antes de salvar o dado bruto.")

caminho_bruto_clima = (
    PASTA_RAW / f"nasa_power_{config['local'].replace('-', '_')}"
    f"_{config['ano_inicial']}_{config['ano_final']}.json"
)

with open(caminho_bruto_clima, "w", encoding="utf-8") as f:
    json.dump(dados_clima_agricola, f, ensure_ascii=False, indent=2)

print("Resposta bruta preservada em:", caminho_bruto_clima)

Resposta bruta preservada em: /home/joaoguilherme/Desktop/bx_produtividade_agricola/data/raw/nasa_power_Piracicaba_SP_2015_2025.json


## 5. Coleta 2 — Produtividade agrícola (IBGE, tabela 1612 da PAM)

### 5.1 Confirmar o código do município

In [5]:
url_localidades = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/SP/municipios"

try:
    resposta_localidades = requests.get(url_localidades, timeout=30)
    resposta_localidades.raise_for_status()
    print("Requisição concluída.")
except requests.RequestException as erro:
    print("Falha na requisição:", erro)

municipios_sp = resposta_localidades.json()

nome_municipio = config["local"].split("-")[0].strip()
municipio_encontrado = next(
    (m for m in municipios_sp if m["nome"] == nome_municipio), None
)

if municipio_encontrado is None:
    raise ValueError(f"Município '{nome_municipio}' não encontrado na lista de SP.")

codigo_municipio = municipio_encontrado["id"]
print("Código IBGE do município:", codigo_municipio)

Requisição concluída.
Código IBGE do município: 3538709


### 5.2 Consultar os metadados do agregado 1612

Buscamos nos metadados a classificação "Produto das lavouras temporárias" e, dentro dela, a categoria cujo nome corresponde à cultura definida em `config`.

In [6]:
codigo_agregado = 1612  # PAM: área plantada, área colhida, quantidade produzida,
                          # rendimento médio e valor da produção das lavouras temporárias

url_metadados_ibge = f"https://servicodados.ibge.gov.br/api/v3/agregados/{codigo_agregado}/metadados"

resposta_meta = requests.get(url_metadados_ibge, timeout=30)
resposta_meta.raise_for_status()
metadados_ibge = resposta_meta.json()

classificacao_produto = next(
    c for c in metadados_ibge["classificacoes"]
    if "lavouras temporárias" in c["nome"].lower()
)

categoria_cultura = next(
    cat for cat in classificacao_produto["categorias"]
    if config["cultura"].split(" ")[0].lower() in cat["nome"].lower()
)

codigo_classificacao = classificacao_produto["id"]
codigo_cultura = categoria_cultura["id"]

print("Classificação:", classificacao_produto["nome"], "-> id", codigo_classificacao)
print("Categoria:", categoria_cultura["nome"], "-> id", codigo_cultura)

print("\nVariáveis disponíveis no agregado:")
for var in metadados_ibge["variaveis"]:
    print(" ", var["id"], "-", var["nome"])

Classificação: Produto das lavouras temporárias -> id 81
Categoria: Soja (em grão) -> id 2713

Variáveis disponíveis no agregado:
  109 - Área plantada
  1000109 - Área plantada - percentual do total geral
  216 - Área colhida
  1000216 - Área colhida - percentual do total geral
  214 - Quantidade produzida
  112 - Rendimento médio da produção
  215 - Valor da produção
  1000215 - Valor da produção - percentual do total geral


### 5.3 Selecionar somente as variáveis justificadas

As mesmas quatro variáveis já documentadas no dicionário de dados: rendimento médio (alvo), área plantada, área colhida e quantidade produzida.

In [ ]:
nomes_variaveis_interesse = [
    "Área plantada",
    "Área colhida",
    "Quantidade produzida",
    "Rendimento médio da produção",
]

variaveis_selecionadas = {
    var["nome"]: var["id"]
    for var in metadados_ibge["variaveis"]
    if any(nome in var["nome"] for nome in nomes_variaveis_interesse)
}

print(variaveis_selecionadas)

codigos_variaveis = "|".join(str(v) for v in variaveis_selecionadas.values())

{'Área plantada': 109, 'Área plantada - percentual do total geral': 1000109, 'Área colhida': 216, 'Área colhida - percentual do total geral': 1000216, 'Quantidade produzida': 214, 'Rendimento médio da produção': 112}


### 5.4 Requisição dos dados agrícolas

In [12]:
if "codigos_variaveis" not in globals():
    nomes_variaveis_interesse = [
        "Área plantada",
        "Área colhida",
        "Quantidade produzida",
        "Rendimento médio da produção",
    ]
    variaveis_selecionadas = {
        var["nome"]: var["id"]
        for var in metadados_ibge["variaveis"]
        if any(nome in var["nome"] for nome in nomes_variaveis_interesse)
    }
    codigos_variaveis = "|".join(str(v) for v in variaveis_selecionadas.values())

periodos = f"{config['ano_inicial']}-{config['ano_final']}"

url_ibge = (
    f"https://servicodados.ibge.gov.br/api/v3/agregados/{codigo_agregado}"
    f"/periodos/{periodos}/variaveis/{codigos_variaveis}"
)

parametros_ibge = {
    "localidades": f"N6[{codigo_municipio}]",
    "classificacao": f"{codigo_classificacao}[{codigo_cultura}]",
}

try:
    resposta_ibge = requests.get(url_ibge, params=parametros_ibge, timeout=30)
    resposta_ibge.raise_for_status()
    print("Requisição concluída.")
except requests.RequestException as erro:
    print("Falha na requisição:", erro)

Requisição concluída.


In [13]:
print("Status:", resposta_ibge.status_code)
print("URL consultada:", resposta_ibge.url)

dados_ibge = resposta_ibge.json()

assert isinstance(dados_ibge, list), "Resposta inesperada: esperava uma lista de variáveis."
print("Quantidade de variáveis retornadas:", len(dados_ibge))
print("Chaves do primeiro elemento:", dados_ibge[0].keys())

Status: 200
URL consultada: https://servicodados.ibge.gov.br/api/v3/agregados/1612/periodos/2015-2025/variaveis/109%7C1000109%7C216%7C1000216%7C214%7C112?localidades=N6%5B3538709%5D&classificacao=81%5B2713%5D
Quantidade de variáveis retornadas: 6
Chaves do primeiro elemento: dict_keys(['id', 'variavel', 'unidade', 'resultados'])


In [14]:
caminho_bruto_ibge = (
    PASTA_RAW / f"ibge_pam_{config['local'].replace('-', '_')}"
    f"_{config['ano_inicial']}_{config['ano_final']}.json"
)

with open(caminho_bruto_ibge, "w", encoding="utf-8") as f:
    json.dump(dados_ibge, f, ensure_ascii=False, indent=2)

print("Resposta bruta preservada em:", caminho_bruto_ibge)


Resposta bruta preservada em: /home/joaoguilherme/Desktop/bx_produtividade_agricola/data/raw/ibge_pam_Piracicaba_SP_2015_2025.json
